In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import time

# Get the directory where this notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent  # Go up one level to project root
src_path = project_root / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cs_priors.plotting.plotting import (
    plot_overview,
    wrapper_plot_geometry,
    plot_equation,
    plot_matrices,
)
from cs_priors.simulation.mixing_model import run_simulation, just_YAX_from_simulation
from correct_sparsity import heatmap_average_wrong_sources, vector_wrong_predictions, noise_threshold
from cs_priors.solvers.vectorized_sparse_prior import sparse_prior_solution
from cs_priors.solvers.complex_lasso import complex_lasso


# 0. Hvilket oppsett jobber vi med

In [ ]:
num_mics = 100
num_sources = 100
s_sparse = 2
seed = 0
angle_step = 2 * np.pi / num_sources

sim = run_simulation(num_mics=num_mics, num_sources=num_sources, s_sparse=s_sparse, seed=seed, angle_step=angle_step)
X0 = np.linalg.pinv(sim.A[:,:, 1]) @ sim.Y[:, 1]
tol = noise_threshold(X0)
wrapper_plot_geometry(sim, skip_labels=True)
vector_wrong_predictions(sim.X[:, 1], X0, tol=tol, plot=True)

# 1.  Oppdager modellen hvilke kilder som er aktive

In [ ]:
source_range = [num_sources]
# define as percentages of num_sources
mic_range = [int(p * source_range[0]) for p in [0.1, 0.2, 0.9, 1.0]]
sparsity_range = [1, 2, 3]
debug= False
max_iter = 20000
seeds = 11

# run in parallel
# Parallel(n_jobs=-1)([
#     heatmap_average_wrong_sources(lambda Y, A: np.linalg.pinv(A) @ Y, mic_range=mic_range, source_range=source_range, sparsity_range=sparsity_range, name="Pseudoinverse", debug=debug, seeds=seeds),
#     heatmap_average_wrong_sources(lambda Y, A: sparse_prior_solution(Y, A, max_iter=max_iter), mic_range=mic_range, source_range=source_range, sparsity_range=sparsity_range, name="Sparse Prior", debug=debug, seeds=seeds),
#     heatmap_average_wrong_sources(lambda Y, A: complex_lasso(Y, A, alpha=1e-4, max_iter=max_iter), mic_range=mic_range, source_range=source_range, sparsity_range=sparsity_range, name="Complex Lasso", debug=debug, seeds=seeds)
# ])   

# time the runs sequentially

time_start = time.time()

heatmap_average_wrong_sources(lambda Y, A: np.linalg.pinv(A) @ Y, mic_range=mic_range, source_range=source_range, sparsity_range=sparsity_range, name="Pseudoinverse", debug=debug, seeds=seeds)
time_x0 = time.time()
print(f"Time for Pseudoinverse: {time_x0 - time_start:.2f} seconds")

heatmap_average_wrong_sources(lambda Y, A: sparse_prior_solution(Y, A, max_iter=max_iter), mic_range=mic_range, source_range=source_range, sparsity_range=sparsity_range, name="Sparse Prior", debug=debug, seeds=seeds)
time_sp = time.time()
print(f"Time for Sparse Prior: {time_sp - time_x0:.2f} seconds")

heatmap_average_wrong_sources(lambda Y, A: complex_lasso(Y, A, alpha=1e-4, max_iter=max_iter), mic_range=mic_range, source_range=source_range, sparsity_range=sparsity_range, name="Complex Lasso", debug=debug, seeds=seeds)
time_end = time.time()
print(f"Time for Complex Lasso: {time_end - time_sp:.2f} seconds")

# Group LASSO